In [ ]:
# -------------------------
# Stress & Sleep Wellness Analyzer (Final Polished, Pastel UI)
# Paste this entire cell into one Jupyter Notebook cell and run.
# Requirements: pandas, numpy, matplotlib, scikit-learn, ipywidgets, joblib
# Install (if needed): !pip install pandas numpy matplotlib scikit-learn ipywidgets joblib
# -------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report

# ---------- CONFIG ----------
DATA_PATH = "dataset/sleep.csv"   # change if your CSV is elsewhere
RANDOM_STATE = 42
KMEANS_CLUSTERS = 3

# ---------- helper: pastel CSS for nicer UI ----------
display(HTML("""
<style>
/* pastel card */
.pastel-card {
  background: linear-gradient(135deg, #fff6fb 0%, #f2fbff 100%);
  border-radius: 14px;
  padding: 14px;
  box-shadow: 0 6px 18px rgba(46, 49, 74, 0.06);
  border: 1px solid rgba(200,200,210,0.35);
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial;
}
.pastel-title { font-size: 20px; font-weight:600; color:#2c2e43; margin-bottom:6px; }
.pastel-sub { font-size:13px; color:#5b5f73; margin-bottom:10px; }
.small-muted { font-size:11px; color:#7c8092; }
</style>
"""))

# ---------- 1) Load dataset safely ----------
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Put your CSV there or change DATA_PATH.")

# normalize column names
df.columns = df.columns.str.strip()

# quick info
print(f"Loaded dataset: {DATA_PATH} | shape = {df.shape}")
display(df.head(3))

# ---------- 2) Basic cleaning ----------
# fill object NaNs with 'None' and strip
cat_cols = [c for c in df.columns if df[c].dtype == "object"]
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip().fillna("None")

# numeric missing -> median fill
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols:
    if df[c].isnull().sum() > 0:
        df[c] = df[c].fillna(df[c].median())

# required targets
if 'Stress Level' not in df.columns or 'Quality of Sleep' not in df.columns:
    raise ValueError("Dataset must contain columns 'Stress Level' and 'Quality of Sleep'.")

# ---------- 3) Encoding categorical variables ----------
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# ---------- 4) Feature engineering ----------
if 'Sleep Duration' in df.columns:
    df['Sleep_Efficiency'] = df['Sleep Duration'] / 24.0
else:
    df['Sleep_Efficiency'] = 0.0

if 'Daily Steps' in df.columns and 'Age' in df.columns:
    df['Activity_Ratio'] = df['Daily Steps'] / (df['Age'] + 1)
else:
    df['Activity_Ratio'] = 0.0

# Drop ID columns if present
drop_cols = []
if 'Person ID' in df.columns:
    drop_cols.append('Person ID')

# Prepare features and targets
X = df.drop(columns=drop_cols + ['Stress Level', 'Quality of Sleep'])
y_stress = df['Stress Level']
y_sleep = df['Quality of Sleep']

print("Using feature columns:", list(X.columns))

# ---------- 5) Clustering for insight ----------
scaler_for_clust = StandardScaler()
X_scaled_full = scaler_for_clust.fit_transform(X)
kmeans = KMeans(n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE)
df['Cluster'] = kmeans.fit_predict(X_scaled_full)

print("Cluster distribution:")
display(df['Cluster'].value_counts())

# small visual (one chart)
plt.figure(figsize=(6,3.5))
if 'Sleep Duration' in df.columns:
    plt.scatter(df['Sleep Duration'], df['Stress Level'], c=df['Cluster'])
    plt.xlabel('Sleep Duration')
    plt.ylabel('Stress Level')
    plt.title('Lifestyle clusters (Sleep Duration vs Stress Level)')
    plt.tight_layout()
    plt.show()

# ---------- 6) Train/test split and scalers (separate splits) ----------
# Stress split
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y_stress, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y_stress if len(np.unique(y_stress))>1 else None
)
scaler_s = StandardScaler()
X_train_s_scaled = scaler_s.fit_transform(X_train_s)
X_test_s_scaled = scaler_s.transform(X_test_s)

# Sleep split
X_train_sl, X_test_sl, y_train_sl, y_test_sl = train_test_split(
    X, y_sleep, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y_sleep if len(np.unique(y_sleep))>1 else None
)
scaler_sl = StandardScaler()
X_train_sl_scaled = scaler_sl.fit_transform(X_train_sl)
X_test_sl_scaled = scaler_sl.transform(X_test_sl)

# ---------- 7) Train models ----------
# Stress (MLP)
mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=RANDOM_STATE)
mlp.fit(X_train_s_scaled, y_train_s)
y_pred_s = mlp.predict(X_test_s_scaled)
print("\nStress model performance:")
print("Accuracy:", accuracy_score(y_test_s, y_pred_s))
print(classification_report(y_test_s, y_pred_s, zero_division=0))

# Sleep (RandomForest)
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced')
rf.fit(X_train_sl_scaled, y_train_sl)
y_pred_sl = rf.predict(X_test_sl_scaled)
print("\nSleep model performance:")
print("Accuracy:", accuracy_score(y_test_sl, y_pred_sl))
print(classification_report(y_test_sl, y_pred_sl, zero_division=0))

# feature importance (sleep model)
try:
    feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nTop features for Sleep model:")
    display(feat_imp.head(8))
    plt.figure(figsize=(7,2.8))
    feat_imp.head(8).plot(kind='bar')
    plt.title('Top features (Sleep model)')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Feature importance error:", e)

# save models + scalers (files in notebook working dir)
joblib.dump(mlp, "stress_model_jupyter.pkl")
joblib.dump(rf, "sleep_model_jupyter.pkl")
joblib.dump(scaler_s, "scaler_stress_jupyter.pkl")
joblib.dump(scaler_sl, "scaler_sleep_jupyter.pkl")
joblib.dump(le_dict, "label_encoders_jupyter.pkl")
print("\nModels and encoders saved to current directory.")

# ---------- Recommendations function ----------
def suggest_improvement_from_vals(sleep_hours, stress_level_val, physical_activity_level, daily_steps):
    advice = []
    # sleep advice
    if sleep_hours < 6:
        advice.append("Try to increase sleep to 7–8 hours and keep a consistent bedtime.")
    elif sleep_hours >= 9:
        advice.append("Sleeping 9+ hours occasionally may indicate poor sleep quality — monitor energy and routines.")
    else:
        advice.append("Sleep hours look reasonable — keep a steady schedule.")
    # stress advice (percentile based)
    p75 = np.percentile(df['Stress Level'], 75)
    p50 = np.percentile(df['Stress Level'], 50)
    if stress_level_val >= p75:
        advice.append("High stress — try short mindfulness breaks, deep breathing, or talk to someone you trust.")
    elif stress_level_val >= p50:
        advice.append("Moderate stress — maintain breaks, exercise, and avoid late-night screens.")
    else:
        advice.append("Low stress — keep maintaining your balance!")
    # activity advice
    if physical_activity_level < 3 or daily_steps < 5000:
        advice.append("Increase activity: aim for 30 mins daily and 7–8k steps per day.")
    else:
        advice.append("Good activity — consistency matters.")
    return advice

# ---------- 8) Beautiful onboarding + pastel UI with ipywidgets ----------
# Onboarding screen (name input) - appears first
name_input = widgets.Text(value='', placeholder='Type your name...', description='', layout=widgets.Layout(width='60%'))
welcome_label = widgets.HTML("<div class='pastel-card'><div class='pastel-title'>Hey there 🌿</div>"
                             "<div class='pastel-sub'>What's your name? We'll personalize the wellness check.</div></div>")
continue_btn = widgets.Button(description="Continue", button_style='success', layout=widgets.Layout(width='160px'))
onboard_out = widgets.Output()

# Main UI widgets (hidden until continue)
age_widget = widgets.IntSlider(description='Age', min=18, max=80, value=25, layout=widgets.Layout(width='420px'))
sleep_widget = widgets.FloatSlider(description='Sleep hrs', min=2.0, max=12.0, step=0.5, value=7.0, layout=widgets.Layout(width='420px'))
activity_widget = widgets.IntSlider(description='Phys Activity (1-10)', min=1, max=10, value=5, layout=widgets.Layout(width='420px'))
steps_widget = widgets.IntText(description='Daily Steps', value=5000, layout=widgets.Layout(width='200px'))

# create dropdowns from encoder classes (if available) with fallbacks
def classes_or_default(col, default):
    if col in le_dict:
        classes = list(le_dict[col].classes_)
        return classes if len(classes)>0 else default
    return default

gender_widget = widgets.Dropdown(description='Gender', options=classes_or_default('Gender',['Male','Female','Other']), value=classes_or_default('Gender',['Male','Female','Other'])[0])
occupation_widget = widgets.Dropdown(description='Occupation', options=classes_or_default('Occupation',['Student','Professional','Other']), value=classes_or_default('Occupation',['Student','Professional','Other'])[0])
bmi_widget = widgets.Dropdown(description='BMI', options=classes_or_default('BMI Category',['Underweight','Normal','Overweight','Obese']), value=classes_or_default('BMI Category',['Underweight','Normal','Overweight','Obese'])[0])
bp_widget = widgets.Dropdown(description='Blood Pressure', options=classes_or_default('Blood Pressure',['Low','Normal','High']), value=classes_or_default('Blood Pressure',['Low','Normal','High'])[0])
sleep_disorder_widget = widgets.Dropdown(description='Sleep Disorder', options=classes_or_default('Sleep Disorder',['None','Insomnia','Apnea','Other']), value=classes_or_default('Sleep Disorder',['None','Insomnia','Apnea','Other'])[0])

analyze_btn = widgets.Button(description='Analyze', button_style='primary', layout=widgets.Layout(width='160px'))
reset_btn = widgets.Button(description='Reset', button_style='', layout=widgets.Layout(width='120px'))
ui_out = widgets.Output()

# Container boxes
onboard_box = widgets.VBox([welcome_label, name_input, continue_btn, onboard_out], layout=widgets.Layout(width='520px'))
inputs_box = widgets.VBox([
    widgets.HTML("<div class='pastel-card'><div class='pastel-title'>Personal details</div>"
                 "<div class='pastel-sub'>Fill these to get a personalized analysis.</div></div>"),
    widgets.HBox([age_widget, sleep_widget]),
    widgets.HBox([activity_widget, steps_widget]),
    widgets.HBox([gender_widget, occupation_widget, bmi_widget]),
    widgets.HBox([bp_widget, sleep_disorder_widget]),
    widgets.HBox([analyze_btn, reset_btn]),
], layout=widgets.Layout(width='520px'))

main_container = widgets.VBox([inputs_box, ui_out], layout=widgets.Layout(padding='10px', border='0px solid #ddd'))

# Hide main container initially
main_container.layout.display = 'none'

# ---------- event handlers ----------
def on_continue_clicked(b):
    name = name_input.value.strip()
    if len(name) == 0:
        with onboard_out:
            clear_output()
            print("Please enter your name to continue 🙂")
        return
    # hide onboarding, show main UI
    onboard_box.layout.display = 'none'
    main_container.layout.display = None
    with ui_out:
        clear_output()
        display(HTML(f"<div class='pastel-card'><div class='pastel-title'>Hey {name} 🌞</div>"
                     "<div class='pastel-sub'>Let's check your stress & sleep — enter values and click Analyze.</div></div>"))

def on_reset_clicked(b):
    # reset inputs to defaults
    age_widget.value = 25
    sleep_widget.value = 7.0
    activity_widget.value = 5
    steps_widget.value = 5000
    # reset selects to first option
    gender_widget.value = gender_widget.options[0]
    occupation_widget.value = occupation_widget.options[0]
    bmi_widget.value = bmi_widget.options[0]
    bp_widget.value = bp_widget.options[0]
    sleep_disorder_widget.value = sleep_disorder_widget.options[0]
    with ui_out:
        clear_output()
        print("Inputs reset. Adjust values and analyze again.")

def on_analyze_clicked(b):
    with ui_out:
        clear_output()
        # Build input row matching X columns
        try:
            input_dict = {}
            for col in X.columns:
                # numeric straightforward mapping
                if col == 'Age':
                    input_dict[col] = age_widget.value
                elif col == 'Sleep Duration':
                    input_dict[col] = sleep_widget.value
                elif col == 'Physical Activity Level':
                    input_dict[col] = activity_widget.value
                elif col == 'Daily Steps':
                    input_dict[col] = steps_widget.value
                elif col == 'Sleep_Efficiency':
                    input_dict[col] = sleep_widget.value / 24.0
                elif col == 'Activity_Ratio':
                    input_dict[col] = steps_widget.value / (age_widget.value + 1)
                else:
                    # categorical: pick the proper encoder and transform selected label
                    if col in le_dict:
                        le = le_dict[col]
                        # heuristics for mapping widget to column
                        if 'gender' in col.lower():
                            raw = gender_widget.value
                        elif 'occup' in col.lower():
                            raw = occupation_widget.value
                        elif 'bmi' in col.lower():
                            raw = bmi_widget.value
                        elif 'pressure' in col.lower() or 'blood' in col.lower():
                            raw = bp_widget.value
                        elif 'sleep disorder' in col.lower():
                            raw = sleep_disorder_widget.value
                        else:
                            # default fallback: try widgets matching name
                            raw = gender_widget.value
                        # transform safely
                        try:
                            enc = int(le.transform([raw])[0])
                        except Exception:
                            # if label not found, fallback to 0
                            enc = 0
                        input_dict[col] = enc
                    else:
                        input_dict[col] = 0

            input_df = pd.DataFrame([input_dict], columns=X.columns)

            # scale for each model
            input_scaled_s = scaler_s.transform(input_df)
            input_scaled_sl = scaler_sl.transform(input_df)

            # predictions
            stress_pred = mlp.predict(input_scaled_s)[0]
            sleep_pred = rf.predict(input_scaled_sl)[0]

            # greeting name
            name = name_input.value.strip()
            display(HTML(f"<div class='pastel-card'><div class='pastel-title'>Results for {name} 🌿</div>"
                         "<div class='pastel-sub'>Below are your AI predictions and suggestions.</div></div>"))

            # show predictions
            print("🧠 Predictions:")
            print(f" • Predicted Stress Level: {stress_pred}")
            print(f" • Predicted Sleep Quality: {sleep_pred}\n")

            # recommendations
            recs = suggest_improvement_from_vals(sleep_widget.value, stress_pred, activity_widget.value, steps_widget.value)
            display(HTML("<b>💡 Personalized Recommendations:</b>"))
            for r in recs:
                display(HTML(f"<div style='margin-left:6px;'>• {r}</div>"))

            # cluster info
            input_scaled_for_clust = scaler_for_clust.transform(input_df)
            cluster_label = kmeans.predict(input_scaled_for_clust)[0]
            display(HTML(f"<div class='small-muted' style='margin-top:8px;'>Lifestyle cluster: <b>{cluster_label}</b></div>"))
            display(HTML("<hr style='margin-top:8px;margin-bottom:8px;'>"))

            # show model probabilities if available
            if hasattr(mlp, "predict_proba"):
                probs_s = mlp.predict_proba(input_scaled_s)[0]
                labels_s = mlp.classes_
                plt.figure(figsize=(5,2.2))
                plt.bar([str(x) for x in labels_s], probs_s)
                plt.title("Stress model probabilities")
                plt.ylabel("Probability")
                plt.tight_layout()
                plt.show()

            if hasattr(rf, "predict_proba"):
                probs_sl = rf.predict_proba(input_scaled_sl)[0]
                labels_sl = rf.classes_
                plt.figure(figsize=(5,2.2))
                plt.bar([str(x) for x in labels_sl], probs_sl)
                plt.title("Sleep model probabilities")
                plt.ylabel("Probability")
                plt.tight_layout()
                plt.show()

        except Exception as e:
            print("Error during analysis:", e)

# wire event handlers
continue_btn.on_click(on_continue_clicked)
reset_btn.on_click(on_reset_clicked)
analyze_btn.on_click(on_analyze_clicked)

# ---------- show onboarding ----------
display(onboard_box)
display(main_container)

# End of cell
